# Coral Reef Health Detector - YOLOv5 (Client Version)

Run coral disease detection online using your trained YOLOv5 model. No heavy hardware required.

### Step 1: Install Dependencies

In [ ]:
%pip install -q torch torchvision opencv-python-headless ultralytics
print("Dependencies ready!")

### Step 2: Upload Your Test Image
Run this cell and select the coral image you want to analyze.
Make sure your `yolov5_best.pt` is already uploaded to the left sidebar at `/content/yolov5_best.pt`.

In [ ]:
from google.colab import files
print("Please select your test image now:")
uploaded = files.upload()
test_file = list(uploaded.keys())[0] if uploaded else None
if not test_file:
    print("ERROR: No file uploaded!")
else:
    print(f"Uploaded: {test_file}")

### Step 3: Run Analysis

In [ ]:
import torch
import os
from IPython.display import display, Image as IPImage
from google.colab import files

model_path = '/content/yolov5_best.pt'

if not os.path.exists(model_path):
    print(f"ERROR: Model not found at {model_path}. Upload yolov5_best.pt to the sidebar first!")
elif not test_file or not os.path.exists(test_file):
    print("ERROR: No test image found. Run Step 2 first!")
else:
    print(f"Loading YOLOv5 model...")
    model = torch.hub.load('ultralytics/yolov5', 'custom', path=model_path, force_reload=False, trust_repo=True)
    model.conf = 0.25

    print(f"Analyzing '{test_file}'...\n")
    results = model(test_file)

    print("\n" + "="*40)
    print("DETECTION SUMMARY (YOLOv5):")
    print("="*40)
    detections = results.pandas().xyxy[0]
    if detections.empty:
        print("No coral issues detected in this image.")
    else:
        for _, row in detections.iterrows():
            class_name = row['name']
            confidence = row['confidence'] * 100
            if class_name == "Healthy Coral":
                tag = "[Healthy]"
            elif "disease" in class_name.lower() or "pox" in class_name.lower():
                tag = "[Disease]"
            else:
                tag = "[Warning]"
            print(f"{tag:<12} {class_name}: {confidence:.2f}%")
    print("="*40 + "\n")

    # Save and display
    results.save(save_dir='/content/yolo_output')
    out_path = f'/content/yolo_output/{os.path.basename(test_file)}'
    if os.path.exists(out_path):
        display(IPImage(filename=out_path))
        print("Downloading your annotated image...")
        files.download(out_path)
    else:
        imgs = [f for f in os.listdir('/content/yolo_output') if f.endswith(('.jpg','.png','.jpeg'))]
        if imgs:
            final = f'/content/yolo_output/{imgs[-1]}'
            display(IPImage(filename=final))
            files.download(final)
        else:
            print("Could not find output image. Check /content/yolo_output/ manually.")